# 1D Preprocessing

In [1]:
import os
import sys
# Add the parent directory to the Python path to allow module imports
sys.path.append(os.path.dirname(os.getcwd()))

import json
import numpy as np
import pandas as pd
import shutil
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm

from modules.datasets import ICBHIAudioDataset, KAUHAudioDataset
from modules.lungsound import LungSoundAudio
from modules.transforms import *

In [2]:
DATA_PATH = Path(os.path.join(os.path.dirname(os.getcwd()), "data"))
RAW_DATA_FOLDER = DATA_PATH / "raw"
INTERIM_DATA_FOLDER = DATA_PATH / "interim"

if not os.path.exists(RAW_DATA_FOLDER):
    raise FileNotFoundError(f"Raw data folder not found at {RAW_DATA_FOLDER}. Please ensure the original data was already downloaded and placed in the correct location.")

if not os.path.exists(INTERIM_DATA_FOLDER):
    os.makedirs(INTERIM_DATA_FOLDER)
    print(f"Created interim data folder at {INTERIM_DATA_FOLDER}.")
else:
    if len(os.listdir(INTERIM_DATA_FOLDER)) > 0:
        print(f"[WARNING] Interim data folder already exist and is not empty ({INTERIM_DATA_FOLDER}). Consider deleting it to run the preprocessing step again.")

Created interim data folder at /home/leticialopes/Projects/IA901/IA901_Project/data/interim.


## Interim

In [3]:
TARGET_SR = 22050   # Hz
WINDOW_LENGTH = 5.0 # seconds
HOP_LENGTH = 2      # seconds (overlap of 3 seconds)

def preprocess_audios(original_data_path: Path, preprocessed_data_path: Path):
    """
    Preprocesses the original data and saves the preprocessed data to the specified location.
    Args:
        original_data_path (Path): Path to the original raw data.
        preprocessed_data_path (Path): Path where the preprocessed data will be saved.
    """
    # Grab all files in the original data directory, .wav or not, including subdirectories
    all_files = sorted(original_data_path.glob("**/*.*"))
    # Iterate through all files and apply preprocessing to audio files, while copying non-audio files
    wav_count = 0
    for file in tqdm(all_files, desc=f"Preprocessing {original_data_path.name}"):
        if file.suffix.lower() == ".wav":
            wav_count += 1
            # Load the audio file using the LungSound class
            audio = LungSoundAudio(str(file))
            # Apply preprocessing transforms
            # 1. Split the audio into windows of fixed duration
            cropped_audios = Window(window_length=WINDOW_LENGTH, hop_length=HOP_LENGTH)(audio)
            for i, cropped_audio in enumerate(cropped_audios):
                # 2. Resample the audio to the target sampling rate
                resampled_audio = Resample(target_sr=TARGET_SR)(cropped_audio)
                # 3. Normalize the audio
                normalized_audio = NormalizeAudio()(resampled_audio)
                # Save the preprocessed audio to the new location
                start = int(i * HOP_LENGTH)
                end = int(start + WINDOW_LENGTH)
                new_file_name = f"{file.stem}_clip-{start:03d}-{end:03d}.wav"
                relative_path = file.parent.relative_to(original_data_path)
                preprocessed_file_path = preprocessed_data_path / relative_path / new_file_name
                preprocessed_file_path.parent.mkdir(parents=True, exist_ok=True)
                sf.write(preprocessed_file_path, normalized_audio.audio, normalized_audio.sr)

        else:
            # If it's not an audio file, simply copy it to the new location
            relative_path = file.parent.relative_to(original_data_path)
            new_file_path = preprocessed_data_path / relative_path / file.name
            new_file_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(file, new_file_path)

    print(f"Preprocessing completed for '{original_data_path.name}'.")
    print(f"Total audio files processed: {wav_count}")

    # Save a file containing the total number of preprocessed audio files
    audio_transforms = {
        Window.__name__: {"params": vars(Window(window_length=WINDOW_LENGTH, hop_length=HOP_LENGTH))},
        Resample.__name__: {"params": vars(Resample(target_sr=TARGET_SR))},
        NormalizeAudio.__name__: {"params": vars(NormalizeAudio())},
    }
    with open(preprocessed_data_path / "preprocessing.json", "w") as f:
        json.dump({"audio_transforms": audio_transforms}, f, indent=4)

In [4]:
preprocess_audios(RAW_DATA_FOLDER, INTERIM_DATA_FOLDER)

Preprocessing raw:   0%|          | 0/2194 [00:00<?, ?it/s]

Preprocessing completed for 'raw'.
Total audio files processed: 1256
